# 3/ Automated product review and classification with SQL functions


In this demo, we will explore the SQL AI function `ai_query` to create a pipeline extracting product review information.

<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/product/sql-ai-functions/sql-ai-query-function-flow.png" width="1000">

**Don't forget about built-in SQL AI functions!** *In this notebook, we show you how to create your own custom functions. However, many text-related tasks (translation, classification etc.) are available as [builtin SQL functions]($./01-Builtin-SQL-AI-Functions). If you can, prefere these as they're easy to use and performant!*

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=DBSQL&org_id=2162748966026566&notebook=%2F03-automated-product-review-and-answer&demo_name=sql-ai-functions&event=VIEW&path=%2F_dbdemos%2FDBSQL%2Fsql-ai-functions%2F03-automated-product-review-and-answer&version=1">

To run this notebook, connect to <b> SQL endpoint </b>. The AI_QUERY function is available on Databricks SQL Pro and Serverless.

In [0]:
-- as previously, make sure you run this notebook using a SQL Warehouse or Serverless endpoint (not a classic cluster)
SELECT assert_true(current_version().dbsql_version is not null, 'YOU MUST USE A SQL WAREHOUSE OR SERVERLESS, not a classic cluster');

USE CATALOG main;
USE SCHEMA dbdemos_ai_query;

"assert_true((current_version().dbsql_version IS NOT NULL), YOU MUST USE A SQL WAREHOUSE OR SERVERLESS, not a classic cluster)"
null



## Simplifying AI function access for SQL users

As reminder, `ai_query` signature is the following:

```
SELECT ai_query(<Endpoint Name>, <prompt>)
```

In the [previous notebook]($./02-Generate-fake-data-with-AI-functions-Foundation-Model), we created a wrapper `ASK_LLM_MODEL` function to simplify our SQL operation and hide the configuration details to end-users. We will re-use this function for this pipeline.

In order to simplify the user-experience for our analysts, we will build prescriptive SQL functions that ask natural language questions of our data and return the responses as structured data.

In [0]:
SELECT * FROM fake_reviews INNER JOIN fake_customers using (customer_id)

customer_id,review_date,review_id,review,firstname,lastname,order_count
1,2022-01-01,1234567890,"I loved the new Kellogg's cereal, it's crunchy and delicious!",Eleanor,Wiggins,143
2,2022-01-05,2345678901,"The Coca-Cola soda I received was flat, very disappointing.",Lucas,Brooks,67
3,2022-01-10,3456789012,"The Nestle coffee is okay, but the price is a bit high for the quality.",May,Parker,118
4,2022-01-15,4567890123,I've been buying the Pepsi soda for years and it never fails to satisfy my thirst.,Oliver,Russell,191
5,2022-01-20,5678901234,"The General Mills yogurt is a great snack, but the packaging could be more eco-friendly.",Ava,Lee,73
6,2022-01-25,6789012345,"I was really looking forward to trying the new Lay's chips, but they were too spicy for my taste.",Mason,Hall,28
7,2022-02-01,7890123456,"The Pringles can is very convenient, but the flavor is not as good as the original.",Isabella,Walker,165
8,2022-02-05,8901234567,"I've tried many different types of coffee, but the Starbucks coffee is still my favorite.",Logan,Young,51
9,2022-02-10,9012345678,"The Kraft mac and cheese is a classic, but it's not as healthy as I would like.",Charlotte,Allen,138
10,2022-02-15,1023456789,"I was surprised by how much I enjoyed the Dove chocolate, it's rich and creamy.",William,King,82


## Review analysis with prompt engineering 
&nbsp;
The keys to getting useful results back from a LLM model are:
- Asking it a well-formed question
- Being specific about the type of answer that you are expecting

In order to get results in a form that we can easily store in a table, we'll ask the model to return the result in a string that reflects `JSON` representation, and be very specific of the schema that we expect

Here's the prompt we've settled on:
```
A customer left a review on a product. We want to follow up with anyone who appears unhappy.
Extract all entities mentioned. For each entity:
- classify sentiment as ["POSITIVE","NEUTRAL","NEGATIVE"]
- whether customer requires a follow-up: Y or N
- reason for requiring followup

Return JSON ONLY. No other text outside the JSON. JSON format:
[{
    "product_name": <product name>,
    "category": <product category>,
    "sentiment": <review sentiment, one of ["POSITIVE","NEUTRAL","NEGATIVE"]>,
    "followup": <Y or N for follow up>,
    "followup_reason": <reason for followup>
}]

Review:
<insert review text here>
```

In [0]:
CREATE OR REPLACE FUNCTION ANNOTATE_REVIEW(review STRING)
    RETURNS STRUCT<product_name: STRING, entity_sentiment: STRING, followup: STRING, followup_reason: STRING>
    RETURN FROM_JSON(
      ASK_LLM_MODEL(CONCAT(
        'A customer left a review. We follow up with anyone who appears unhappy.
         extract the following information:
          - classify sentiment as ["POSITIVE","NEUTRAL","NEGATIVE"]
          - returns whether customer requires a follow-up: Y or N
          - if followup is required, explain what is the main reason

        Return JSON ONLY. No other text outside the JSON. JSON format:
        {
            "product_name": <entity name>,
            "entity_sentiment": <entity sentiment>,
            "followup": <Y or N for follow up>,
            "followup_reason": <reason for followup>
        }
        
        Review:', review), "{'type': 'json_object'}"),
      "STRUCT<product_name: STRING, entity_sentiment: STRING, followup: STRING, followup_reason: STRING>")

-- ALTER FUNCTION ANNOTATE_REVIEW OWNER TO `your_principal`; -- for the demo only, make sure other users can access your function

In [0]:
CREATE OR REPLACE TABLE reviews_annotated as 
    SELECT * EXCEPT (review_annotated), review_annotated.* FROM (
      SELECT *, ANNOTATE_REVIEW(review) AS review_annotated
        FROM fake_reviews)
    INNER JOIN fake_customers using (customer_id)

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM reviews_annotated

customer_id,review_date,review_id,review,firstname,lastname,order_count,product_name,entity_sentiment,followup,followup_reason
1,2022-01-01,1234567890,"I loved the new Kellogg's cereal, it's crunchy and delicious!",Eleanor,Wiggins,143,Kellogg's cereal,POSITIVE,N,null
2,2022-01-05,2345678901,"The Coca-Cola soda I received was flat, very disappointing.",Lucas,Brooks,67,Coca-Cola,NEGATIVE,Y,"The customer received a flat soda, which was very disappointing."
3,2022-01-10,3456789012,"The Nestle coffee is okay, but the price is a bit high for the quality.",May,Parker,118,Nestle coffee,NEUTRAL,Y,Customer is unhappy with the price-quality ratio
4,2022-01-15,4567890123,I've been buying the Pepsi soda for years and it never fails to satisfy my thirst.,Oliver,Russell,191,Pepsi soda,POSITIVE,N,null
5,2022-01-20,5678901234,"The General Mills yogurt is a great snack, but the packaging could be more eco-friendly.",Ava,Lee,73,General Mills yogurt,NEUTRAL,Y,customer is unhappy with the packaging
6,2022-01-25,6789012345,"I was really looking forward to trying the new Lay's chips, but they were too spicy for my taste.",Mason,Hall,28,Lay's chips,NEGATIVE,Y,The customer found the product too spicy for their taste.
7,2022-02-01,7890123456,"The Pringles can is very convenient, but the flavor is not as good as the original.",Isabella,Walker,165,Pringles,NEGATIVE,Y,Customer is unhappy with the flavor
8,2022-02-05,8901234567,"I've tried many different types of coffee, but the Starbucks coffee is still my favorite.",Logan,Young,51,Starbucks coffee,POSITIVE,N,None
9,2022-02-10,9012345678,"The Kraft mac and cheese is a classic, but it's not as healthy as I would like.",Charlotte,Allen,138,Kraft mac and cheese,NEUTRAL,Y,Customer expressed a concern about the product's healthiness
10,2022-02-15,1023456789,"I was surprised by how much I enjoyed the Dove chocolate, it's rich and creamy.",William,King,82,Dove chocolate,POSITIVE,N,None


In [0]:
CREATE OR REPLACE FUNCTION GENERATE_RESPONSE(firstname STRING, lastname STRING, order_count INT, product_name STRING, reason STRING)
  RETURNS STRING
  RETURN ASK_LLM_MODEL(
    CONCAT("Our customer named ", firstname, " ", lastname, " who ordered ", order_count, " ", product_name, " was unhappy about ", product_name, "specifically due to ", reason, ". Provide an empathetic message I can send to my customer 
    including the offer to have a call with the relevant product manager to leave feedback. I want to win back their 
    favour and I do not want the customer to churn"), 
    "{'type': 'text'}"
  );
-- ALTER FUNCTION GENERATE_RESPONSE OWNER TO `account users`; -- for the demo only, make sure other users can access your function

In [0]:
SELECT GENERATE_RESPONSE("Quentin", "Ambard", 235, "Country Choice Snacking Cookies", "Quality issue") AS customer_response

customer_response
"Here's a message you can send to Quentin Ambard: ""Dear Quentin, I'm so sorry to hear that you were unhappy with the quality of the Country Choice Snacking Cookies you received. We take all complaints seriously and are truly sorry that we didn't meet your expectations. At [Your Company Name], we pride ourselves on providing the best possible products and service to our customers, and it's clear that we fell short in your case. I want to assure you that we're committed to making things right and ensuring that you have a positive experience with us. I'd like to invite you to speak with our Product Manager, who would love to hear your feedback directly. Your input is invaluable in helping us to identify areas for improvement and make necessary changes to our products. Would you be available for a quick call at your convenience? This will give us the opportunity to listen to your concerns and work together to find a solution. Please know that we value your business and would like to make things right. If there's anything else we can do to rectify the situation, please don't hesitate to let us know. Your satisfaction is our top priority, and we're committed to winning back your trust. If you're interested in speaking with our Product Manager, please let me know a time that suits you, and I'll arrange the call. Alternatively, if you'd prefer to provide feedback via email, you can reply to this message and I'll ensure that your comments are passed on to the relevant team. Once again, I apologize for the disappointment caused, and I look forward to the opportunity to make things right. Best regards, [Your Name]"" This message aims to: * Acknowledge the customer's disappointment and apologize for the issue * Show empathy and a commitment to making things right * Offer a solution (a call with the Product Manager) to gather feedback and work towards a resolution * Reassure the customer that their business is valued and that you're committed to winning back their trust * Provide a clear call to action (scheduling a call or providing feedback via email) to encourage the customer to engage with you further."


In [0]:
CREATE OR REPLACE TABLE reviews_answer as 
    SELECT *,
      generate_response(firstname, lastname, order_count, product_name, followup_reason) AS response_draft
    FROM reviews_annotated where followup='Y'

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM reviews_answer

customer_id,review_date,review_id,review,firstname,lastname,order_count,product_name,entity_sentiment,followup,followup_reason,response_draft
2,2022-01-05,2345678901,"The Coca-Cola soda I received was flat, very disappointing.",Lucas,Brooks,67,Coca-Cola,NEGATIVE,Y,"The customer received a flat soda, which was very disappointing.","Here's a message you can send to Lucas Brooks: ""Dear Lucas, I am so sorry to hear that your recent experience with our Coca-Cola order was disappointing. Receiving a flat soda is not the level of quality we strive to provide, and I can understand why it would be frustrating. I want to start by apologizing for the inconvenience this has caused and assure you that we take situations like this very seriously. We value your business and appreciate the trust you've placed in us by ordering from us. I'd like to make things right and ensure that you have a better experience with us in the future. I'd like to offer you the opportunity to speak with our Product Manager, who would love to hear your feedback directly. Your input will help us to identify areas for improvement and make necessary changes to prevent similar issues from happening again. Would you be available for a quick call at your convenience? This will give you a chance to share your thoughts and concerns, and we can also discuss how we can prevent this from happening again in the future. Please let me know if this is something you'd be interested in, and I'll arrange a call at a time that suits you. Your satisfaction is our top priority, and I'm committed to making things right. Thank you for your patience and understanding. I look forward to hearing from you soon. Best regards, [Your Name]"" This message aims to: * Acknowledge the customer's disappointment and apologize for the inconvenience * Show empathy and understanding for the customer's frustration * Offer a solution to make things right and prevent similar issues in the future * Provide an opportunity for the customer to provide feedback and be heard * Demonstrate a commitment to customer satisfaction and loyalty By responding in this way, you can help to win back Lucas Brooks' favor and prevent him from churning."
3,2022-01-10,3456789012,"The Nestle coffee is okay, but the price is a bit high for the quality.",May,Parker,118,Nestle coffee,NEUTRAL,Y,Customer is unhappy with the price-quality ratio,"Here's a message you can send to May Parker: ""Dear May, I'm so sorry to hear that you're not satisfied with your recent purchase of 118 Nestle coffees. I understand that the price-quality ratio is a crucial factor in your purchasing decisions, and I apologize if we fell short of your expectations. I want to assure you that we value your feedback and would like to make things right. I'd like to offer you a personal call with our Product Manager, who would be happy to listen to your concerns and gather your feedback on our Nestle coffee products. Your input will help us to improve our offerings and ensure that we're meeting the high standards you deserve. The call will be a great opportunity for you to share your thoughts and suggestions on how we can better meet your needs. We're committed to providing the best possible products and services, and your feedback is invaluable in helping us achieve this goal. If you're available, please let me know a convenient time for the call, and I'll make sure to arrange it with our Product Manager. Alternatively, if you'd prefer to provide your feedback via email, I'm more than happy to receive it and pass it on to the relevant team. As a valued customer, we appreciate your loyalty and would like to ensure that you're completely satisfied with your purchases from us. We're willing to work with you to find a solution that meets your needs and exceeds your expectations. Please know that we're committed to winning back your trust and ensuring that you continue to choose us for your coffee needs. If there's anything else I can do to make things right, please 

### Creating our Customer Review Dashboard

The next step is to set up a comprehensive dashboard in order to track and monitor our customer reviews.

Open the <a dbdemos-dashboard-id="customer-review-analysis" href='/sql/dashboardsv3/01f1b2d4d0be1ca3bb9f960ff06be97f' target="_blank">Customer review analysis dashboard</a> to have a complete view of your customers, products and reviews.

<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/product/sql-ai-functions/sql-ai-function-dashboard.png" width="1200">


### Going further

Our pipeline is ready. Keep in mind that this is a fairly basic pipeline for our demo.

For more advanced pipeline, we recommend using Spark Declarative Pipelines. SDP simplify data ingetsion and transformation tasks with incremental load, materialized view and more advanced features. For more details, run `dbdemos.install_demo('pipeline-bike')`

### Extra: AdHoc Queries

Remember that analysts can always use the `ASK_LLM_MODEL()` function we created earlier to apply their own prompts to the data.

As short example, let's write a query to extract all review about beverages:

In [0]:
SELECT review_id,
    BOOLEAN(ASK_LLM_MODEL(
      CONCAT("Does this review discuss beverages? Answer boolean: 'true' or 'false' only, lowercase, no explanations or notes nor final dot. Review: ", review)
    )) AS is_beverage_review,
    review
  FROM fake_reviews LIMIT 10

review_id,is_beverage_review,review
1234567890,false,"I loved the new Kellogg's cereal, it's crunchy and delicious!"
2345678901,true,"The Coca-Cola soda I received was flat, very disappointing."
3456789012,true,"The Nestle coffee is okay, but the price is a bit high for the quality."
4567890123,true,I've been buying the Pepsi soda for years and it never fails to satisfy my thirst.
5678901234,false,"The General Mills yogurt is a great snack, but the packaging could be more eco-friendly."
6789012345,false,"I was really looking forward to trying the new Lay's chips, but they were too spicy for my taste."
7890123456,false,"The Pringles can is very convenient, but the flavor is not as good as the original."
8901234567,true,"I've tried many different types of coffee, but the Starbucks coffee is still my favorite."
9012345678,false,"The Kraft mac and cheese is a classic, but it's not as healthy as I would like."
1023456789,false,"I was surprised by how much I enjoyed the Dove chocolate, it's rich and creamy."


### Extra: Create an Production Ready Pipeline using Spark Declarative Pipelines

We can turn the steps in this notebook into a production ready Spark Declarative Pipelines pipeline with AI SQL Functions

Open [04-create-end-to-end-DLT-workflow]($./04-create-end-to-end-DLT-workflow) for more details.

## You're now ready to process your text using external LLM models!

We've seen that the lakehouse provide advanced AI capabilities, not only you can leverage external LLM APIs, but you can also build your own LLM with Databricks GenAI applications!
For more details on creating your chatbot with the Lakehouse, run: `dbdemos.install('llm-rag-chatbot')`

Go back to [the introduction]($./01-SQL-AI-Functions-Introduction)

## Extra: configure an External Model Endpoint to leverage external providers (OpenAI, Anthropic...) 

This demo was using one Databricks Foundation Model (pricing token-based).

Your model endpoint can also be setup to use an external model such as OpenAI. Open [05-Extra-setup-external-model-OpenAI]($./05-Extra-setup-external-model-OpenAI) for more details.


Go back to [the introduction]($./00-SQL-AI-Functions-Introduction)